In [2]:
import sys, os
REPO_PATH = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
print(f"REPO_PATH: {REPO_PATH}")  # debug

REPO_PATH: c:\Users\Aymen ich\Projects\ML_pipeline\flipkart_sentiments_pfa


# Flipkart Reviews — Preprocessing NLP
## Auteure : Ihssane Moutchou | PFA 4ème Année
## Objectif : Nettoyer le texte des avis et préparer
## les features TF-IDF pour l'entraînement des modèles.
## Entrée  : data/flipkart_data_with_sentiment.csv (produit par Aymen)
## Sortie  : data/flipkart_data_preprocessed.csv

In [3]:
import pandas as pd
import numpy as np
import re
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import joblib, warnings
warnings.filterwarnings('ignore')
print("Imports OK")

Imports OK


In [5]:
# On charge le CSV que Aymen a produit dans sa Phase 1
# ATTENTION : les sentiments sont 'Positif'/'Négatif'/'Neutre'
# avec majuscule et accents — on va les normaliser
df = pd.read_csv(os.path.join(REPO_PATH, 'data', 'flipkart_data_with_sentiment.csv'))
print(f"Dataset chargé : {df.shape[0]:,} avis")
print("Colonnes :", df.columns.tolist())
print("Sentiments :", df['sentiment'].unique())
print("Valeurs manquantes :\n", df.isnull().sum())
df.head(3)

Dataset chargé : 9,976 avis
Colonnes : ['review', 'rating', 'sentiment', 'review_length', 'lang', 'review_len', 'word_count']
Sentiments : <StringArray>
['Positif', 'Négatif', 'Neutre']
Length: 3, dtype: str
Valeurs manquantes :
 review              0
rating              0
sentiment           0
review_length       0
lang             8976
review_len          0
word_count          0
dtype: int64


,review,rating,sentiment,review_length,lang,review_len,word_count
0,It was nice produt. I like it's design a lot. ...,5,Positif,18,NaN,98,18
1,awesome sound....very pretty to see this nd th...,5,Positif,24,NaN,134,24
2,awesome sound quality. pros 7-8 hrs of battery...,4,Positif,80,NaN,499,80


In [6]:
# Normaliser : 'Positif' → 'POSITIF' etc. pour cohérence
mapping = {
    'Positif': 'POSITIF',
    'Négatif': 'NEGATIF',
    'Neutre':  'NEUTRE'
}
df['sentiment'] = df['sentiment'].map(mapping)

# Supprimer les lignes avec review vide
df = df.dropna(subset=['review'])
df['review'] = df['review'].astype(str)

print("Après normalisation :")
print(df['sentiment'].value_counts())

Après normalisation :
sentiment
POSITIF    8091
NEGATIF    1001
NEUTRE      884
Name: count, dtype: int64


In [7]:
# Pipeline NLP : lowercase → regex → tokenize → stopwords → lemmatize
lemmatizer = WordNetLemmatizer()
stop_words  = set(stopwords.words('english'))

def clean_text(text):
    # 1. Minuscules
    text = text.lower()
    # 2. Supprimer ponctuation et chiffres
    text = re.sub(r'[^a-z\s]', '', text)
    # 3. Tokeniser
    tokens = word_tokenize(text)
    # 4. Supprimer stopwords + mots courts
    tokens = [t for t in tokens
              if t not in stop_words and len(t) > 2]
    # 5. Lemmatiser
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# Test rapide
exemple = "The product is really good! I loved it and bought 2 pieces."
print("Avant :", exemple)
print("Après :", clean_text(exemple))

Avant : The product is really good! I loved it and bought 2 pieces.
Après : product really good loved bought piece


In [8]:
# Appliquer sur tout le dataset (peut prendre 1–2 min)
print("Nettoyage en cours...")
df['review_clean'] = df['review'].apply(clean_text)
print(f"Terminé — {len(df):,} avis nettoyés")

# Vérification : comparer avant/après
for _, row in df.sample(3, random_state=42).iterrows():
    print(f"\n[{row['sentiment']}]")
    print("Avant :", row['review'][:80])
    print("Après :", row['review_clean'][:80])

Nettoyage en cours...
Terminé — 9,976 avis nettoyés

[POSITIF]
Avant : It's good but mic was not working after 2 monthsREAD MORE
Après : good mic working monthsread

[POSITIF]
Avant : Writing review after 2 weeks of use...      Believe me..its totally worth the mo
Après : writing review week use believe meits totally worth money design good good batte

[POSITIF]
Avant : GoodREAD MORE
Après : goodread


In [9]:
# TF-IDF : texte → vecteurs numériques
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

X = tfidf.fit_transform(df['review_clean'])
y = df['sentiment']

print(f"Matrice TF-IDF : {X.shape[0]:,} avis × {X.shape[1]:,} features")

# Split 80% train / 20% test — graine fixe pour reproductibilité
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train : {X_train.shape[0]:,} avis")
print(f"Test  : {X_test.shape[0]:,} avis")

# Distribution des classes dans train
print("\nDistribution train :")
print(pd.Series(y_train).value_counts())

Matrice TF-IDF : 9,976 avis × 5,000 features
Train : 7,980 avis
Test  : 1,996 avis

Distribution train :
sentiment
POSITIF    6472
NEGATIF     801
NEUTRE      707
Name: count, dtype: int64


In [12]:
# Sauvegarder le vectorizer TF-IDF (sera chargé dans les modèles)
joblib.dump(tfidf, os.path.join(REPO_PATH, 'models', 'tfidf_vectorizer.pkl'))
print("tfidf_vectorizer.pkl sauvegardé dans models/")

# Sauvegarder le CSV nettoyé pour Aymen (Phase 3)
df[['review','review_clean','sentiment']].to_csv(
    os.path.join(REPO_PATH, 'data', 'flipkart_data_preprocessed.csv'), index=False
)
print("flipkart_data_preprocessed.csv sauvegardé")

# Résumé final
print("\n"+"="*50)
print("SYNTHÈSE PREPROCESSING")
print("="*50)
print(f"Avis traités     : {len(df):,}")
print(f"Features TF-IDF  : 5 000")
print(f"Train / Test     : {X_train.shape[0]:,} / {X_test.shape[0]:,}")
print("Preprocessing terminé — prêt pour la Phase 3 !")

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (1065255645.py, line 7)

In [ ]:
import joblib
import pandas as pd

# Sauvegarder les données d'entraînement et de test
joblib.dump(X_train, '/content/drive/MyDrive/flipkart_sentiments_pfa/data/X_train.pkl')
joblib.dump(X_test, '/content/drive/MyDrive/flipkart_sentiments_pfa/data/X_test.pkl')
y_train.to_csv('/content/drive/MyDrive/flipkart_sentiments_pfa/data/y_train.csv', index=False)
y_test.to_csv('/content/drive/MyDrive/flipkart_sentiments_pfa/data/y_test.csv', index=False)

print("X_train.pkl, X_test.pkl, y_train.csv, y_test.csv sauvegardés dans data/")